# MalariAI — Phase 3: EfficientNet-B0 + Focal Loss (Kaggle)

**Kaysarul Anas Apurba · Laurentian University**  
Pipeline B · Stage 2 · EfficientNet-B0 · NIH BBBC041 · 30 epochs

---

### Session guide
| Step | Do this |
|------|---------|
| **First run** | Set `RESUME_FROM = None`, run all cells top to bottom |
| **Resume** | Upload `last.pth` as a Kaggle dataset, set `RESUME_FROM` to its path, run all |

### Why crops are pre-extracted (Cell 4)
Loading full 1600×1200 images for every crop makes each epoch ~40 min.  
Pre-extracting 64×64 crops once (Cell 4, ~10 min) drops each epoch to ~3-5 min.  
Cell 4 is skipped automatically if crops already exist.

## 1 · Configuration — edit here only

In [1]:
from pathlib import Path

# ── Paths — update to match your Kaggle dataset slug ──────────────────────
DATASET_ROOT = Path('/kaggle/input/datasets/kaysarulanas/bbbc041-malaria/malaria')
# ──────────────────────────────────────────────────────────────────────────

TRAIN_JSON   = DATASET_ROOT / 'training.json'
IMG_DIR      = DATASET_ROOT / 'images'

# Working directories (Kaggle /kaggle/working persists for the session)
OUT_DIR      = Path('/kaggle/working/phase3_checkpoints')
CROPS_DIR    = Path('/kaggle/working/crops')   # pre-extracted 64x64 crops

OUT_DIR.mkdir(parents=True, exist_ok=True)
CROPS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
EPOCHS      = 30
BATCH_SIZE  = 128   # larger batch is fine now that crops are tiny files
LR          = 1e-4
VAL_FRAC    = 0.15
NUM_WORKERS = 4
CROP_SIZE   = 64
GAMMA       = 2.0   # Focal Loss focusing exponent

# Resume — set to path of last.pth from a previous Kaggle session
# Example: '/kaggle/input/phase3-checkpoint/last.pth'
RESUME_FROM = None

print(f'Train JSON : {TRAIN_JSON}')
print(f'Images     : {IMG_DIR}')
print(f'Crops dir  : {CROPS_DIR}')
print(f'Output dir : {OUT_DIR}')
assert TRAIN_JSON.exists(), f'training.json not found at {TRAIN_JSON}'
assert IMG_DIR.exists(),    f'images dir not found at {IMG_DIR}'
print('Paths OK.')


Train JSON : /kaggle/input/datasets/kaysarulanas/bbbc041-malaria/malaria/training.json
Images     : /kaggle/input/datasets/kaysarulanas/bbbc041-malaria/malaria/images
Crops dir  : /kaggle/working/crops
Output dir : /kaggle/working/phase3_checkpoints
Paths OK.


## 2 · GPU Check

In [2]:
import subprocess, sys, time
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'No GPU')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
PyTorch 2.10.0+cu128  |  CUDA: True
Device : Tesla T4
VRAM   : 15.6 GB


## 3 · Imports

In [3]:
import json, time, csv, sys
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torchvision.transforms as T
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')


Using: cuda


## 4 · Label Map

In [4]:
LABEL_TO_INT = {
    'background':     0,
    'red blood cell': 1,
    'trophozoite':    2,
    'ring':           3,
    'schizont':       4,
    'gametocyte':     5,
    'leukocyte':      6,
}
INT_TO_LABEL    = {v: k for k, v in LABEL_TO_INT.items()}
NUM_CLASSES     = 7
PARASITE_NAMES  = ['trophozoite', 'ring', 'schizont', 'gametocyte']
SKIP_LABELS     = {'difficult'}

# Training annotation counts from Phase 1 EDA (excluding 'difficult')
TRAIN_COUNTS = {1: 77420, 2: 1473, 3: 353, 4: 179, 5: 144, 6: 103}

print('Label map loaded.')
for i, name in INT_TO_LABEL.items():
    cnt = TRAIN_COUNTS.get(i, 0)
    print(f'  [{i}] {name:<22}  count={cnt:>6}')


Label map loaded.
  [0] background              count=     0
  [1] red blood cell          count= 77420
  [2] trophozoite             count=  1473
  [3] ring                    count=   353
  [4] schizont                count=   179
  [5] gametocyte              count=   144
  [6] leukocyte               count=   103


## 5 · Pre-extract Crops (one-time, ~10 min)

**Why:** Loading full 1600×1200 images for every crop makes each epoch ~40 min.  
This cell loads each image **once**, extracts all its crops, saves 64×64 PNGs.  
After this, each epoch loads tiny files → ~3-5 min per epoch.

**Skip:** If `crops/manifest.csv` already exists this cell does nothing.

In [5]:
import sys

manifest_path = CROPS_DIR / 'manifest.csv'

if manifest_path.exists():
    with open(manifest_path) as f:
        n = sum(1 for _ in f) - 1
    print(f'Crops already extracted ({n} crops). Skipping.')
else:
    print('Pre-extracting crops — loading each image once ...')
    t0 = time.time()

    # Parse JSON and group annotations by image
    with open(TRAIN_JSON) as f:
        raw = json.load(f)

    # image_path -> list of (x1,y1,x2,y2, label_idx)
    image_crops = defaultdict(list)
    for item in raw:
        fname    = item['image']['pathname'].lstrip('/')
        img_path = IMG_DIR / fname
        if not img_path.exists():
            img_path = IMG_DIR / Path(fname).name
        if not img_path.exists():
            continue
        for ann in item.get('objects', []):
            lbl = ann.get('category', '')
            if lbl in SKIP_LABELS or lbl not in LABEL_TO_INT:
                continue
            bb = ann.get('bounding_box', {})
            x1 = int(bb.get('minimum', {}).get('c', 0))
            y1 = int(bb.get('minimum', {}).get('r', 0))
            x2 = int(bb.get('maximum', {}).get('c', 0))
            y2 = int(bb.get('maximum', {}).get('r', 0))
            if x2 > x1 and y2 > y1:
                image_crops[str(img_path)].append(
                    (x1, y1, x2, y2, LABEL_TO_INT[lbl]))

    print(f'  {len(image_crops)} unique images to process')
    total_crops = 0

    with open(manifest_path, 'w', newline='') as csvf:
        writer = csv.writer(csvf)
        writer.writerow(['crop_file', 'label'])

        for img_idx, (img_path_str, crops) in enumerate(image_crops.items()):
            img    = Image.open(img_path_str).convert('RGB')
            stem   = Path(img_path_str).stem

            for crop_idx, (x1, y1, x2, y2, label_idx) in enumerate(crops):
                crop = img.crop((x1, y1, x2, y2)).resize(
                    (CROP_SIZE, CROP_SIZE), Image.BILINEAR)
                fname = f'{stem}_{crop_idx:04d}.png'
                crop.save(str(CROPS_DIR / fname))
                writer.writerow([fname, label_idx])
                total_crops += 1

            if (img_idx + 1) % 100 == 0:
                elapsed = time.time() - t0
                pct = (img_idx+1) / len(image_crops) * 100
                eta = elapsed / (img_idx+1) * (len(image_crops) - img_idx - 1)
                print(f'  [{img_idx+1}/{len(image_crops)}] '
                      f'{pct:.0f}%  crops={total_crops}  '
                      f'elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m',
                      flush=True)

    elapsed = time.time() - t0
    print(f'\nDone. {total_crops} crops saved in {elapsed/60:.1f} min')
    print(f'Manifest: {manifest_path}')

# Print class distribution
counts = defaultdict(int)
with open(manifest_path) as f:
    reader = csv.DictReader(f)
    for row in reader:
        counts[int(row['label'])] += 1
print('\nClass distribution in crops:')
for i in range(1, NUM_CLASSES):
    print(f'  [{i}] {INT_TO_LABEL[i]:<22}: {counts.get(i,0):>6}')


Pre-extracting crops — loading each image once ...
  1208 unique images to process
  [100/1208] 8%  crops=6136  elapsed=0.3m  eta=3.1m
  [200/1208] 17%  crops=12981  elapsed=0.5m  eta=2.7m
  [300/1208] 25%  crops=19745  elapsed=0.8m  eta=2.4m
  [400/1208] 33%  crops=25629  elapsed=1.0m  eta=2.1m
  [500/1208] 41%  crops=31997  elapsed=1.3m  eta=1.8m
  [600/1208] 50%  crops=38351  elapsed=1.5m  eta=1.5m
  [700/1208] 58%  crops=44638  elapsed=1.7m  eta=1.3m
  [800/1208] 66%  crops=51713  elapsed=2.0m  eta=1.0m
  [900/1208] 75%  crops=58126  elapsed=2.3m  eta=0.8m
  [1000/1208] 83%  crops=65159  elapsed=2.5m  eta=0.5m
  [1100/1208] 91%  crops=72194  elapsed=2.8m  eta=0.3m
  [1200/1208] 99%  crops=79166  elapsed=3.0m  eta=0.0m

Done. 79672 crops saved in 3.1 min
Manifest: /kaggle/working/crops/manifest.csv

Class distribution in crops:
  [1] red blood cell        :  77420
  [2] trophozoite           :   1473
  [3] ring                  :    353
  [4] schizont              :    179
  [5] gam

## 6 · Dataset (loads pre-extracted 64×64 crops)

In [6]:
class CropDataset(Dataset):
    """Fast dataset — loads pre-extracted 64x64 PNG crops from disk."""

    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, manifest_path, crops_dir, train=True):
        self.crops_dir = Path(crops_dir)
        self.items     = []   # list of (filename, label_idx)

        aug = [
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(15),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        ] if train else []

        self.transform = T.Compose([
            *aug,
            T.ToTensor(),
            T.Normalize(self.MEAN, self.STD),
        ])

        with open(manifest_path) as f:
            for row in csv.DictReader(f):
                self.items.append((row['crop_file'], int(row['label'])))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        fname, label = self.items[idx]
        img = Image.open(self.crops_dir / fname).convert('RGB')
        return self.transform(img), label


# Build datasets
print('Building datasets from pre-extracted crops ...')
full_ds  = CropDataset(manifest_path, CROPS_DIR, train=True)
n_val    = max(1, int(len(full_ds) * VAL_FRAC))
n_train  = len(full_ds) - n_val

train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(42))

# Val split: no augmentation
val_ds.dataset = CropDataset(manifest_path, CROPS_DIR, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True)

print(f'Train crops : {n_train}')
print(f'Val crops   : {n_val}')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')


Building datasets from pre-extracted crops ...
Train crops : 67722
Val crops   : 11950
Train batches: 530  |  Val batches: 94


## 7 · Focal Loss

In [7]:
def compute_focal_alpha():
    total  = sum(TRAIN_COUNTS.values())
    alpha  = [0.0]  # background weight = 0
    for i in range(1, NUM_CLASSES):
        cnt = TRAIN_COUNTS.get(i, 1)
        alpha.append(total / (len(TRAIN_COUNTS) * cnt))
    s     = sum(alpha[1:])
    alpha = [a / s * (NUM_CLASSES - 1) for a in alpha]
    return torch.tensor(alpha, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)
        self.gamma = gamma

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, reduction='none')
        pt   = torch.exp(-ce)
        at   = self.alpha[targets]
        return (at * (1.0 - pt) ** self.gamma * ce).mean()


alpha     = compute_focal_alpha().to(device)
criterion = FocalLoss(alpha=alpha, gamma=GAMMA)

print('Focal Loss alpha weights:')
for i in range(NUM_CLASSES):
    print(f'  [{i}] {INT_TO_LABEL[i]:<22} α={alpha[i].item():.4f}')


Focal Loss alpha weights:
  [0] background             α=0.0000
  [1] red blood cell         α=0.0030
  [2] trophozoite            α=0.1581
  [3] ring                   α=0.6597
  [4] schizont               α=1.3010
  [5] gametocyte             α=1.6172
  [6] leukocyte              α=2.2610


## 8 · Model (EfficientNet-B0)

In [8]:
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total:,} total  |  {trainable:,} trainable')


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 208MB/s]

Parameters: 4,016,515 total  |  4,016,515 trainable


## 9 · Resume from previous checkpoint (optional)

In [9]:
start_epoch  = 0
best_val_acc = 0.0
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

if RESUME_FROM and Path(RESUME_FROM).exists():
    print(f'Loading checkpoint: {RESUME_FROM}')
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch  = ckpt['epoch']
    best_val_acc = ckpt.get('best_val_acc', 0.0)
    train_losses = ckpt.get('train_losses', [])
    val_losses   = ckpt.get('val_losses',   [])
    train_accs   = ckpt.get('train_accs',   [])
    val_accs     = ckpt.get('val_accs',     [])
    # Fast-forward LR scheduler to correct state
    for _ in range(start_epoch):
        scheduler.step()
    print(f'Resuming from epoch {start_epoch + 1}  '
          f'(best val acc so far: {best_val_acc*100:.2f}%)')
else:
    print('Starting fresh.')


Starting fresh.


## 10 · Training & Validation Functions

In [10]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for crops, labels in loader:
        crops, labels = crops.to(device), labels.to(device)
        logits = model(crops)
        loss   = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    cls_correct = defaultdict(int)
    cls_total   = defaultdict(int)

    for crops, labels in loader:
        crops, labels = crops.to(device), labels.to(device)
        logits = model(crops)
        loss   = criterion(logits, labels)
        preds  = logits.argmax(1)

        total_loss += loss.item() * len(labels)
        correct    += (preds == labels).sum().item()
        total      += len(labels)

        for lbl, pred in zip(labels.cpu(), preds.cpu()):
            cls_total[lbl.item()]   += 1
            cls_correct[lbl.item()] += int(pred == lbl)

    per_class = {
        INT_TO_LABEL[i]: round(cls_correct[i]/cls_total[i]*100, 2)
        for i in range(1, NUM_CLASSES) if cls_total[i] > 0
    }
    return total_loss / total, correct / total, per_class

print('Functions defined.')


Functions defined.


## 11 · Training Loop

In [11]:
print(f'Training epochs {start_epoch+1} → {EPOCHS}')
print(f'Batch: {BATCH_SIZE}  LR: {LR}  Focal γ: {GAMMA}')
print('-' * 85)

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    tr_loss, tr_acc            = train_one_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, per_class = evaluate(model, val_loader, criterion)
    scheduler.step()

    train_losses.append(tr_loss);  val_losses.append(vl_loss)
    train_accs.append(tr_acc);     val_accs.append(vl_acc)

    elapsed = time.time() - t0
    is_best = vl_acc > best_val_acc

    print(f'Epoch [{epoch+1:03d}/{EPOCHS}]  '
          f'tr_loss={tr_loss:.4f}  vl_loss={vl_loss:.4f}  '
          f'tr_acc={tr_acc*100:.2f}%  vl_acc={vl_acc*100:.2f}%  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed:.0f}s)',
          flush=True)

    if is_best:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), OUT_DIR / 'best.pth')
        cls_str = '  '.join(f'{k[:5]}={v}%' for k,v in per_class.items())
        print(f'  *** Best val acc: {best_val_acc*100:.2f}% — saved best.pth ***', flush=True)
        print(f'  Per-class: {cls_str}', flush=True)

    # Save last.pth every epoch — safe to resume from even if session crashes
    torch.save({
        'epoch':        epoch + 1,
        'model':        model.state_dict(),
        'optimizer':    optimizer.state_dict(),
        'scheduler':    scheduler.state_dict(),
        'best_val_acc': best_val_acc,
        'train_losses': train_losses,
        'val_losses':   val_losses,
        'train_accs':   train_accs,
        'val_accs':     val_accs,
        'cfg': {'epochs': EPOCHS, 'batch': BATCH_SIZE,
                'lr': LR, 'gamma': GAMMA},
    }, OUT_DIR / 'last.pth')

print('\nTraining complete.')


Training epochs 1 → 30
Batch: 128  LR: 0.0001  Focal γ: 2.0
-------------------------------------------------------------------------------------
Epoch [001/30]  tr_loss=0.0135  vl_loss=0.0087  tr_acc=61.88%  vl_acc=68.13%  lr=9.97e-05  (50s)
  *** Best val acc: 68.13% — saved best.pth ***
  Per-class: red b=68.53%  troph=47.51%  ring=63.49%  schiz=62.5%  gamet=70.83%  leuko=86.67%
Epoch [002/30]  tr_loss=0.0069  vl_loss=0.0063  tr_acc=84.04%  vl_acc=83.63%  lr=9.89e-05  (47s)
  *** Best val acc: 83.63% — saved best.pth ***
  Per-class: red b=83.96%  troph=75.57%  ring=55.56%  schiz=79.17%  gamet=75.0%  leuko=86.67%
Epoch [003/30]  tr_loss=0.0055  vl_loss=0.0046  tr_acc=88.52%  vl_acc=87.27%  lr=9.76e-05  (47s)
  *** Best val acc: 87.27% — saved best.pth ***
  Per-class: red b=87.75%  troph=67.87%  ring=73.02%  schiz=66.67%  gamet=83.33%  leuko=100.0%
Epoch [004/30]  tr_loss=0.0043  vl_loss=0.0040  tr_acc=90.51%  vl_acc=91.49%  lr=9.57e-05  (47s)
  *** Best val acc: 91.49% — saved best

## 12 · Final Evaluation (best checkpoint)

In [12]:
print('Loading best.pth ...')
model.load_state_dict(torch.load(OUT_DIR / 'best.pth', map_location=device))
_, final_acc, per_class = evaluate(model, val_loader, criterion)

print(f'\nFinal Val Accuracy: {final_acc*100:.2f}%')
print('Per-class accuracy:')
for cls, acc in sorted(per_class.items(), key=lambda x: -x[1]):
    print(f'  {cls:<22}: {acc:.2f}%')

metrics = {
    'val_accuracy_pct':   round(final_acc * 100, 2),
    'best_val_acc_pct':   round(best_val_acc * 100, 2),
    'total_epochs':       EPOCHS,
    'per_class_accuracy': per_class,
    'train_losses':       [round(x,6) for x in train_losses],
    'val_losses':         [round(x,6) for x in val_losses],
    'train_accs':         [round(x,4) for x in train_accs],
    'val_accs':           [round(x,4) for x in val_accs],
}
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'metrics.json saved.')


Loading best.pth ...

Final Val Accuracy: 98.36%
Per-class accuracy:
  leukocyte             : 100.00%
  red blood cell        : 98.82%
  schizont              : 87.50%
  trophozoite           : 83.71%
  ring                  : 77.78%
  gametocyte            : 75.00%
metrics.json saved.


## 13 · Loss & Accuracy Curves

In [13]:
eps = list(range(1, len(train_losses)+1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(eps, train_losses, 'b-o', ms=3, label='Train')
axes[0].plot(eps, val_losses,   'r-o', ms=3, label='Val')
axes[0].set(xlabel='Epoch', ylabel='Focal Loss',
            title='EfficientNet-B0 — Focal Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(eps, [a*100 for a in train_accs], 'b-o', ms=3, label='Train')
axes[1].plot(eps, [a*100 for a in val_accs],   'r-o', ms=3, label='Val')
axes[1].set(xlabel='Epoch', ylabel='Accuracy (%)',
            title='EfficientNet-B0 — Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'loss_curves.png'), dpi=120)
plt.show()
print('loss_curves.png saved.')


loss_curves.png saved.


## 14 · Download & Resume Guide

### After this session — download these files:

| File | Size | Use |
|------|------|-----|
| `phase3_checkpoints/best.pth` | ~20 MB | Inference + paper results |
| `phase3_checkpoints/last.pth` | ~20 MB | Resume next session |
| `phase3_checkpoints/metrics.json` | tiny | Paper Table 4 |
| `phase3_checkpoints/loss_curves.png` | tiny | Paper Figure |

### To resume in a new Kaggle session:
1. Go to your Kaggle profile → Datasets → New Dataset
2. Upload `last.pth` → name it e.g. `phase3-checkpoint`
3. Add it to this notebook as an input dataset
4. Set `RESUME_FROM = '/kaggle/input/phase3-checkpoint/last.pth'` in Cell 1
5. Run All

In [14]:
# List output files with sizes
print('Files saved to', OUT_DIR)
for f in sorted(OUT_DIR.glob('*')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<35} {size_mb:.1f} MB')


Files saved to /kaggle/working/phase3_checkpoints
  best.pth                            15.6 MB
  last.pth                            46.4 MB
  loss_curves.png                     0.1 MB
  metrics.json                        0.0 MB
